# 02. Shared rows

**Tier:** Intermediate. **Why you'd use this:** You need a grid of zeros for a game board. `[ [0] * cols ] * rows` looks like a grid and makes every row the same list object.

**After this lesson you can:** build independent rows with a comprehension; explain why multiplying a list of lists shares rows.
About 25 minutes. **Uses:** [Lists](../04-collections/01-lists-lesson.ipynb), [Copying](../04-collections/06-copying-lesson.ipynb)


## Quick review

1. Why is a mutable default list shared across calls?
2. What does a shallow copy of a grid still share?

<details><summary>Answers</summary>

The default object is stored once on the function. Nested row lists.
</details>


## The idea

`[row] * n` repeats references to the same `row` object. Multiplying a list of lists therefore builds many names for one inner list. Changing one "row" changes them all. Build each row separately with a comprehension so each inner list is new.

<details><summary>Explain more</summary>

`[0] * cols` is fine for a row of immutable ints. The bug is repeating that row object. `[[0] * cols for _ in range(rows)]` makes a new inner list each pass.
</details>


## See it work

Build a grid with independent rows.


In [1]:
def make_grid(rows, cols):
    # 1. New inner list on every outer pass.
    return [[0] * cols for _ in range(rows)]


grid = make_grid(2, 3)
grid[0][0] = 9
print(grid)
print(grid[0] is grid[1])

[[9, 0, 0], [0, 0, 0]]
False


## Check yourself

Answer these before you run the cell.

1. After `bad[0][0] = 1`, what is `bad[1][0]`?
2. Are `bad[0]` and `bad[1]` the same object?


In [2]:
bad = [[0] * 3] * 2
bad[0][0] = 1
print(bad)
print(bad[0] is bad[1])

[[1, 0, 0], [1, 0, 0]]
True


<details><summary>Answers</summary>

`1`. Yes. Both indexes point at the one inner list.
</details>


## Why it works this way

List multiplication copies references, not deep structure. `[[0] * cols] * rows` evaluates the inner list once, then repeats that pointer `rows` times. Every outer slot holds the same object id. Assignment into any row mutates that one list, so every row appears to change. Deep dive: [Mutability and aliasing](../../internals/02-mutability/01-aliasing-lesson.ipynb). Related lesson: [Copying](../04-collections/06-copying-lesson.ipynb).

A comprehension runs its expression once per outer iteration. Each pass builds a new `[0] * cols` list. The outer list then holds distinct row ids. Immutable cell values such as `0` can still be repeated inside a row safely, because changing a cell replaces an int binding rather than mutating the int.

Cost of the correct grid is one outer list plus `rows` inner lists. The buggy form saves allocations and loses correctness. Evidence is `grid[0] is grid[1]` printing `True` for the multiply form and `False` for the comprehension.

This is the same aliasing family as shallow copies. Multiplying rows is how the bug usually arrives when someone reaches for a short grid literal. If you already have a bad grid, `copy.copy` will not save you, because it still shares the inner rows. Build new rows, or deep-copy, when independence matters. The mechanism is the same pointer repetition you already saw with assignment and shallow copies. Multiplying a nested list is just the shortest way to write that mistake. Print `id` of each row once when a grid looks haunted. Same ids mean you built aliases, not a board.


In [3]:
bad = [[0] * 3] * 2
good = [[0] * 3 for _ in range(2)]
print(bad[0] is bad[1])
print(good[0] is good[1])
bad[0][0] = 1
good[0][0] = 1
print(bad)
print(good)

True
False
[[1, 0, 0], [1, 0, 0]]
[[1, 0, 0], [0, 0, 0]]


## When to use it

Use a comprehension (or a loop that appends new rows) for grids. Use `[0] * n` only for a single row of immutables. Don't multiply a list that contains other lists when you need independence. If any later code mutates a cell, independent rows are required from the start.


## Common mistake

Building a board with `[[0] * cols] * rows`, then wondering why one move fills a whole column of rows.


In [4]:
bad = [[0] * 3] * 2
bad[0][0] = 1
print(bad)

[[1, 0, 0], [1, 0, 0]]


Build each row in a comprehension so the rows do not share identity.


In [5]:
good = [[0] * 3 for _ in range(2)]
good[0][0] = 1
print(good)

[[1, 0, 0], [0, 0, 0]]


## Try it

Change `make_grid` so each cell starts as `None` instead of `0`. Does the same comprehension shape still keep rows independent, and why is `[None] * cols` still safe inside each row?


## Recap

- List multiplication repeats references.
- A grid needs a new inner list per row.
- `is` between rows reveals the bug.

**Next:** [practice](02-shared-rows-practice.ipynb)
